Do imports

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool,OpenAIChatCompletionsModel, RunContextWrapper,trace
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
from openai import AsyncOpenAI
from agents import ModelSettings,RunConfig
from agents import set_trace_processors
import json
import xml.etree.ElementTree as ET
import base64
from openai import OpenAI
from IPython.display import display, Markdown

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    BatchSpanProcessor,
    ConsoleSpanExporter,
)

In [ ]:
provider = TracerProvider()

processor = BatchSpanProcessor(ConsoleSpanExporter())

provider.add_span_processor(processor)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer(__name__)

In [ ]:
load_dotenv(override=True)

SILICONFLOW_API_KEY=os.getenv("SILICONFLOW_API_KEY")
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
#set_trace_processors([])  # Disables all trace exporters
OPENAI_API_KEY=os.getenv("OPENAI_API_KEY") # only for logging, not for model calls

In [ ]:
deepseek_client = AsyncOpenAI(base_url="https://api.deepseek.com", api_key=DEEPSEEK_API_KEY)
deepseek_model_flash = OpenAIChatCompletionsModel(openai_client=deepseek_client,model="deepseek-v4-flash")
deepseek_model_pro = OpenAIChatCompletionsModel(openai_client=deepseek_client,model="deepseek-v4-pro")

In [ ]:
silicon_client = AsyncOpenAI(base_url="https://api.siliconflow.com/v1", api_key=SILICONFLOW_API_KEY)
qwen_vision_model = OpenAIChatCompletionsModel(openai_client=silicon_client,model="Qwen/Qwen3.5-397B-A17B")


* Define_XML_Reader Agent 

In [ ]:
instruct_xml_reader = """
You are an expert process engineer. You are given the content of an XML file exported from a UniSim simulation. Your task is to thoroughly analyze the file and produce a clear, concise description of the process. The description must include the role of each piece of equipment, the role of each stream (material and energy), and **the current numerical values** of key operating parameters (temperatures, pressures, flow rates, compositions, duties, pressure drops, etc.) as they appear in the XML. The output must be a single plain‑text string (no JSON, arrays, or code). Use natural language, but embed the numbers in the text, e.g., "Stream X leaves at 45 °C and 200 kPa with a molar flow of 0.5 kgmole/s."

### What to Include

1. **For each material stream**:
   - Name (exactly as in XML)
   - Source and destination equipment (or "Battery Limit" if it enters/leaves the simulation)
   - Purpose (feed, product, intermediate, recycle, utility, etc.)
   - **Numerical values**: temperature (°C), pressure (kPa), molar flow (kgmole/s) or mass flow (kg/s), and, if available, composition (mole fractions or mass fractions – list major components). If a value is not present, state "not specified".

2. **For each energy stream**:
   - Name
   - Associated equipment
   - Role (heating duty, cooling duty, shaft power)
   - **Numerical value**: duty or power (kJ/s) if available; otherwise state "not specified".

3. **For each piece of equipment**:
   - Name and type (e.g., Distillation Column, Compressor, Cooler)
   - Specific function in the process
   - Key inlet/outlet streams
   - **Numerical values**: relevant parameters (e.g., pressure drop, delta T, efficiency, UA, number of stages, temperature setpoints, etc.) as given in the XML. For columns, include top/bottom pressures and temperatures, reflux ratio, etc.

4. **Overall process summary**: A concise narrative explaining the flow path from feed(s) to product(s), including the purpose of key units and streams.

### Rules for Clarity and Completeness

- Use the exact names as they appear in the XML – do not rename.
- Present the information in a logical order: either by grouping equipment with its connected streams, or by listing all streams first then equipment; either way, cover every object.
- For numerical values, be precise: include units and the value as given (no rounding unless necessary).
- Keep the text readable; use bullet points, numbered lists, or paragraphs as you see fit, but everything must be one continuous string.
- Do not include XML tags, code, or commentary about the extraction process – only the process description.

### Example Output (partial, to illustrate formatting)
Process Description:

Feed stream "Inlet Gas" enters the simulation from outside (battery limit input) and flows to mixer "Saturate". It has T=29.44 °C, P=6205.28 kPa, molar flow=0.1384 kgmole/s, composition (mole fractions): Methane 0.8989, CO2 0.0284, H2S 0.0155, etc.

Stream "Water to Saturate" also enters the battery limit as a utility stream, with T=277.1 °C  – actually the XML shows 277.1 °C, P not specified, molar flow=0.0001386 kgmole/s, pure water.

Mixer "Saturate" (type Mixer) combines Inlet Gas and Water to Saturate to produce stream "Gas + H2O". The outlet stream has T=29.44 °C, P=6205.28 kPa, molar flow=0.13849 kgmole/s.

... [continue for all streams and equipment] ...

Now, analyze the provided XML and output the descriptive string accordingly.
"""

In [ ]:
"""
Vision Analysis Agent
- Reads a local image and converts it to base64
- Uses SiliconFlow + Qwen/Qwen3.5-397B-A17B for analysis
- Built with OpenAI Agents SDK
"""

import asyncio
import base64
import io
import os
from pathlib import Path

from openai import AsyncOpenAI
from PIL import Image
from dotenv import load_dotenv
from agents import (
    Agent,
    ModelProvider,
    OpenAIChatCompletionsModel,
    RunConfig,
    Runner,
    function_tool,
    set_tracing_disabled,
)
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
load_dotenv(override=True)
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")
if not SILICONFLOW_API_KEY:
    raise ValueError("Please set the SILICONFLOW_API_KEY environment variable.")

SILICONFLOW_BASE_URL = "https://api.siliconflow.com/v1"
MODEL_NAME = "Qwen/Qwen3.5-397B-A17B"
from openai import OpenAI          # ← synchronous client
from openai import AsyncOpenAI     # still needed for the agent

# ------------------------------------------------------------------
# Clients
# ------------------------------------------------------------------
# Synchronous client for the tool (important!)
sync_client = OpenAI(
    api_key=SILICONFLOW_API_KEY,
    base_url=SILICONFLOW_BASE_URL,
)

# Async client for the Agents SDK
async_client = AsyncOpenAI(
    api_key=SILICONFLOW_API_KEY,
    base_url=SILICONFLOW_BASE_URL,
)

class SiliconFlowProvider(ModelProvider):
    def get_model(self, model_name: str | None = None):
        return OpenAIChatCompletionsModel(
            model=model_name or MODEL_NAME,
            openai_client=async_client,
        )

SILICONFLOW_PROVIDER = SiliconFlowProvider()


# ------------------------------------------------------------------
# Tool (uses the synchronous client)
# ------------------------------------------------------------------

@function_tool
def analyze_local_image(
    image_path: str,
    question: str = "Describe this image in detail.",
    detail: str = "high",
) -> str:
    """
    Read a local image, convert it to base64, and analyze it with
    Qwen/Qwen3.5-397B-A17B on SiliconFlow.
    """
    path = Path(image_path).expanduser().resolve()
    print(f"[DEBUG] Trying to open: {path}")          # ← visible in Jupyter

    if not path.exists():
        return f"Error: file not found → {path}"
    if not path.is_file():
        return f"Error: path is not a file → {path}"

    try:
        with Image.open(path) as img:
            print(f"[DEBUG] Image opened successfully: {img.size}, mode={img.mode}")
            if img.mode in ("RGBA", "P"):
                img = img.convert("RGB")
            buffer = io.BytesIO()
            img.save(buffer, format="WEBP", quality=90)
            b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
            print(f"[DEBUG] Base64 length: {len(b64)}")

        data_url = f"data:image/webp;base64,{b64}"

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": data_url,
                            "detail": detail,
                        },
                    },
                    {
                        "type": "text",
                        "text": question,
                    },
                ],
            }
        ]

        print("[DEBUG] Calling SiliconFlow...")
        response = sync_client.chat.completions.create(
            model=MODEL_NAME,
            messages=messages,  # type: ignore[arg-type]
            max_tokens=4096,
        )

        content = response.choices[0].message.content
        print("[DEBUG] Got response from model")
        return content or "No content returned."

    except Exception as e:
        import traceback
        error_msg = f"Error: {type(e).__name__}: {e}\n\n{traceback.format_exc()}"
        print(error_msg)          # ← this will show the real problem in the notebook
        return error_msg

# ------------------------------------------------------------------
# Agent
# ------------------------------------------------------------------
vision_agent = Agent(
    name="Vision Analyst",
    instructions="""
You are a helpful vision analysis agent.

When the user gives you a path to an image (or asks you to analyze one):
1. Call the analyze_local_image tool with the correct path and question.
2. Present the analysis clearly to the user.
3. You may add extra observations or answer follow-up questions based on the result.

Always use the tool for any real image analysis. Never invent visual details.
""",
    model=MODEL_NAME,
    tools=[analyze_local_image],
)


# ------------------------------------------------------------------
# Helper for Jupyter
# ------------------------------------------------------------------
async def analyze_image(image_path: str, question: str = "Describe this image in detail.") -> str:
    result = await Runner.run(
        vision_agent,
        f"Please analyze the image at: {image_path}\n\nQuestion: {question}",
        run_config=RunConfig(model_provider=SILICONFLOW_PROVIDER),
    )
    return result.final_output

In [ ]:
# ============================================================
# SERVICES
# ============================================================

class XMLService:
    """
    Responsible only for XML handling.
    """

    def __init__(self, xml_path: str):
        self.xml_path = xml_path


    def load_xml(self) -> str:

        tree = ET.parse(self.xml_path)

        root = tree.getroot()

        return ET.tostring(
            root,
            encoding="unicode"
        )



class VisionService:

    def __init__(
        self,
        image_path,
        client,
        model
    ):

        self.image_path = image_path
        self.client = client
        self.model = model


    def encode_image(self):

        with open(
            self.image_path,
            "rb"
        ) as f:

            return base64.b64encode(
                f.read()
            ).decode("utf-8")


    async def analyze(
        self,
        question,
        model=None,
    ):

        image_base64 = self.encode_image()


        response = self.client.chat.completions.create(

            model=model or self.model,

            messages=[
                {
                    "role": "user",
                    "content":[

                        {
                            "type":"text",
                            "text":question
                        },

                        {
                            "type":"image_url",

                            "image_url":{
                                "url":
                                f"data:image/jpeg;base64,{image_base64}"
                            }
                        }

                    ]
                }
            ]
        )


        return response.choices[0].message.content



In [ ]:
# ============================================================
# APPLICATION CONTEXT
# ============================================================

class ApplicationContext:
    """
    Dependency container.

    It does not know filenames.
    It only stores service objects.
    """

    def __init__(
        self,
        xml_service: XMLService,
        vision_service: VisionService,
    ):

        self.xml_service = xml_service

        self.vision_service = vision_service





# ============================================================
# TOOLS
# ============================================================


@function_tool
def load_xml(
    ctx: RunContextWrapper[ApplicationContext]
):
    """
    Loads the process XML model.
    """

    return ctx.context.xml_service.load_xml()



@function_tool
async def analyze_pfd(
    ctx: RunContextWrapper[ApplicationContext],
    question: str,
    model: str ="Qwen/Qwen3.5-397B-A17B"
):
    """
    Analyzes PFD/P&ID images using a vision model.
    """

    return await ctx.context.vision_service.analyze(
        question,
        model=model,
    )




In [ ]:
optimization_instructions = """
Role: You are a Lead Process Optimization Engineer with 15+ years of experience in steady-state simulation (UniSim Design / HYSYS). 
Your specialty is translating textual process data into a clear, rigorous, and executable mathematical optimization specification for any chemical process.

Input Context:
You will receive two reports. Base your entire analysis **only** on these two reports.
- Report 1 (XML Data Report): Authoritative source for all numerical values, base-case operating points, and **exact literal string names** of streams, energy streams, and equipment as they exist in the UniSim COM interface.
- Report 2 (PFD Context Report): Provides process layout, connectivity, operational objectives, safety/quality constraints, and high-level goals. Use it for context and deriving objectives/constraints.

**Strict Prioritization Rule:**
- For **exact object names** (spelling, spaces, hyphens, capitalization) → ALWAYS use Report 1.
- For numerical base-case values → ALWAYS use Report 1.
- If Report 2 suggests a name or value that conflicts with Report 1, flag the discrepancy and use Report 1’s version.
- Only if a user query remains unanswered by both provided reports, invoke the XML_reader tool exactly once to delegate XML interpretation; do not call it otherwise.

Your Task:
Analyze both reports and produce a complete, unambiguous optimization problem specification that will be handed to a Python script generator (using scipy.optimize or similar) for UniSim COM automation.

Workflow & Critical Instructions:

1. Data Synthesis & Gap Analysis:
   - Extract all relevant base-case values from Report 1.
   - Cross-reference connectivity and objectives from Report 2.
   - Explicitly flag any discrepancies (name mismatches, missing data, conflicting values) in the Data Integrity Check section.
   - If a logical/safety constraint is implied but not explicitly stated, label it clearly as an "Assumption" with reasoning.

2. MANDATORY: Zero-Tolerance for String Alteration (Literal Object Tracking)
   - UniSim COM is extremely sensitive to exact string matches.
   - You MUST preserve the EXACT casing, spaces, hyphens, and punctuation from Report 1.
   - Never sanitize, normalize, remove hyphens, or convert names.

3. Define the Optimization Problem Components:
   - Objective: Choose one clear scalar objective (minimize energy/utility consumption, maximize throughput/yield/profit, maximize product purity, etc.).
   - Decision Variables: Select 2–8 meaningful adjustable variables with clear process impact (e.g., pressures, temperatures, flow rates, split ratios, etc.).
   - Constraints: Include all important operational, safety, quality, equipment, and capacity limits (mostly inequalities).

4. Numerical Stability Guidance:
   - Provide realistic lower/upper bounds based on Report 1 and sound engineering judgment.
   - Suggest appropriate solver tolerances (objective and constraint).
   - Do NOT include any scaling factors or scaled variables in the specification.

5. Algorithm Recommendation:
   - Recommend a suitable algorithm (SLSQP, trust-constr, COBYLA, Differential Evolution, etc.) and justify based on problem size, non-linearity, and simulation call cost.

Output Format (Follow Exactly):

# Optimization Problem Specification

## 1. Data Integrity Check
- **Discrepancies Found:** [List any name/value conflicts or missing information. State "None detected." if clean.]
- **Assumptions Made:** [List any inferred constraints/objectives with clear reasoning.]

## 2. Process Overview
[2-4 sentence summary of the process flow, key unit operations, and the primary optimization goal based on Report 2.]

## 3. Objective Function
**Mathematical Definition:** [Clear equation using variable names, e.g. Minimize Total_Energy = Duty_E1 + Duty_E2 + Duty_E3]
**Units:** [e.g. kJ/s, kg/s, or USD/s]
**Rationale:** [Why this objective aligns with the process goals described in Report 2.]

## 4. Decision Variables
| Variable Name (Python-friendly) | Literal UniSim Object Name (EXACT) | Type | Lower Bound | Upper Bound | Nominal Value (Report 1) | Description & Source |
|--------------------------------|------------------------------------|------|-------------|-------------|---------------------------|----------------------|
| P_intermediate                | "Mid-1"                           | Continuous | 150 kPa    | 1200 kPa   | 450 kPa                  | Pressure of intermediate stream (Report 1) |
| T_reactor                     | "Reactor-Out"                     | Continuous | 80 °C      | 250 °C     | 180 °C                   | Reactor outlet temperature setpoint (Report 1) |

## 5. Constraints (Inequalities)
| Constraint ID | Literal UniSim Tag (EXACT) | Mathematical Expression | Limit | Description & Source |
|---------------|----------------------------|--------------------------|-------|----------------------|
| Purity_spec  | "Product-Stream"          | Product_Purity          | >= 98.5 | Minimum product purity requirement (Report 2) |
| Duty_limit   | "Cooler-E1"               | Cooler_Duty             | <= 850 | Maximum allowable cooler duty (Report 1) |
| Flow_balance | "Feed-Stream"             | Feed_Flow - Min_Required_Flow | >= 0 | Minimum feed flow requirement (Report 1) |

## 6. Optimization Algorithm Recommendation
**Recommended Algorithm:** [e.g., SLSQP]
**Justification:** [Based on number of variables, constraint density, non-linearity, and black-box nature of the simulator.]
**Suggested Termination Criteria:**
- Objective tolerance: 1e-6
- Constraint violation tolerance: 1e-5
- Maximum iterations: 80

CRITICAL NOTES:
- Use ONLY exact string names from Report 1 in the "Literal UniSim Object Name" column.
- Do not invent economic prices or utility costs. Use placeholder [MISSING: REQUIRES ECONOMIC INPUT] if needed and note it in Assumptions.
- If critical data is missing, state it clearly rather than guessing.
- The downstream Python coder will handle implementation details (caching, solver waits, error handling, etc.).

Final Command: Produce a clean, precise, and actionable specification that enables reliable automated optimization for any chemical process.
"""

In [ ]:
# ============================================================
# SPECIALIZED AGENTS
# ============================================================
# ============================================================\n"# SPECIALIZED AGENTS (Defined first without handoffs)
# ============================================================\n
xml_agent = Agent(
    name="XML Parser Agent",
    instructions=instruct_xml_reader,
    model=deepseek_model_flash,
    tools=[load_xml]
)

# MAIN AGENT (Forced Sequential Orchestrator)
# ============================================================
tool1 = xml_agent.as_tool(tool_name="XML_reader", tool_description="Read and extract Process data from xml")

main_agent = Agent(
    name="Process Optimizer Assistant",
    instructions=optimization_instructions,
    model=deepseek_model_pro,
    tools=[tool1]
)


In [ ]:
# --- Instantiate services & context ---
client = OpenAI(base_url="https://api.siliconflow.com/v1", api_key=SILICONFLOW_API_KEY)

xml_service = XMLService(
    xml_path=r".\miniconversion.xml"
)

vision_service = VisionService(
    image_path=r".\pfd.jpg",
    client=client,
    model="Qwen/Qwen3.5-397B-A17B",
)


context = ApplicationContext(
    xml_service=xml_service,
    vision_service=vision_service
)

# --- Orchestration ---


xml_result = await Runner.run(
    xml_agent,
    """Analyze the provided UniSim XML model and return a complete process report with equipment, material streams, energy streams, and a summary.
       You have to specify the role of each piece of equipment and every stream (both material and energy).
        The description must be returned as a single plain-text string, suitable for consumption by another intelligent agent.
        Do not use JSON, arrays, or any machine-readable format – only natural language text.""",
    context=context,
)

print("=== XML Agent Report ===")
print(xml_result.final_output)





In [ ]:

vision_question = """Analyze the provided PFD image and return a complete visual process description with equipment, material streams, energy streams, and a summary.
 Note that blue streams are material flows, while red streams are energy flows. Include the role of each piece of equipment and every stream (both material and energy). The description must be returned as a single plain-text string, suitable for consumption by another intelligent agent. Do not use JSON, arrays, or any machine-readable format – only natural language text."""
vision_text = await analyze_image(
    image_path=r".\pfd.jpg",
    question="Describe this Process flow diagram and all its elements."
)

print("=== Vision Service Report ===")
print(vision_text)

In [ ]:
analysis_prompt = """
You must formulate an objective function to maximize the Venture Profit (VP) of the process. The primary decision variable to adjust in order to maximize this profit is the purge flow rate (W). 
Economic Parameters & Prices:
- Operating Basis: 330 operating days per year (assume 24 hours per day).
- Return on Investment (ROI): 10% (0.1) annually.
- Revenue for ethyl chloride: 2.5 x 10^-3 monetary units/kg

Capital Costs:
The installed cost of equipment depends non-linearly on the mixed (reactor) feed rate FR and is given by the following formula (taken from the textbook Example 21.4):

Installed cost of equipment = 500 * ((330 * 24 / 1000) * FR)^0.6 monetary units

where FR must be the mass flow rate of stream "FR" expressed in kg/hr.
(Do not use the molar flow in kgmole/s. Convert: FR_kg_per_hr = FR_molar * MW_FR * 3600.)

Objective Function Formulation Instructions:
You must derive the Venture Profit (VP) equation in monetary units per year. Generally, it is calculated by taking the annual revenue generated from product P and subtracting the annual raw material costs (based on the feed F and the specific mass fractions of ethylene and HCl). Then, subtract the annualized capital penalty, which is calculated by applying the 10% ROI to the total installed cost of equipment. However, the fresh feed flow rate is constant and cannot be adjusted. Therefore, the feed price is constant and does not affect the optimization.


Constraints:
The maximization of VP is subject to the following constraint:
-	Equipment/Flow Limits: The recycle flow rate must remain strictly below 300 kg/hr (R < 0.0833333 kg/s). Implement it as it is described without relaxation.

"""


Optimization_math_prompt = f"""
{analysis_prompt}

XML report:
{xml_result.final_output}

Vision report:
{vision_text}
"""


result = await Runner.run(
    main_agent,
    Optimization_math_prompt,
    context=context,
    run_config=RunConfig(
                model_settings=ModelSettings(
                    extra_body={
                        "thinking": {"type": "enabled"},
                        "reasoning_effort": "max",      # ← Maximum reasoning effort
                    }
                )
            )
)


In [ ]:


display(Markdown(result.final_output))

In [ ]:
instruct_coder = """
You are an elite senior Python developer specialising in process simulation automation, chemical engineering, and numerical optimisation.
Your task is to generate a flawless, production‑quality, and completely standalone Python script that solves the optimization problem provided in the specification by automating UniSim Design via COM.
Your final output must return the best code that has been run by the tool, the one that produced the best feasible solution among all executed attempts, and you are required to execute every generated code using the run_python_code tool before finalizing, with a strict limit of 5 total execution attempts. After each of the first 4 failed attempts due to errors, nonconvergence, NaN, or infeasibility, you must diagnose the issue from the traceback and diagnostics, then adjust your code by rescaling, changing initial guesses, relaxing bounds, or switching solvers, and rerun. If the 5th attempt also fails, you must still return the best code from among all runs, or the last code if none succeeded, as your result, and under no circumstances may you exceed 5 tool calls.
### CRITICAL RULES (Apply to ANY process):

1. Solver Convergence (Strict):
   - NEVER use `.IsSolved` or `.Solve()`.
   - Use this exact wait loop:
     ```python
     sim.Solver.CanSolve = True
     start = time.time()
     while sim.Solver.IsSolving:
         if time.time() - start > 30:
             raise TimeoutError("Solver timeout")
         time.sleep(0.1)
     ```
   - Fallback: `time.sleep(2.0)` if `IsSolving` is unavailable.

2. Variable Setting (Most Important Reliability Rule):
   - Always modify **MaterialStreams** or **EnergyStreams** properties directly. For example, if you are supposed to modify a pump output temperature, you have to modify the temperature of the stream leaving the pump discharge.
   - Correct pattern for pressures:
     ```python
     fs.MaterialStreams.Item("ExactStreamName").Pressure.SetValue(value, "kPa")
     ```
   - For temperatures:
     ```python
     fs.MaterialStreams.Item("ExactStreamName").Temperature.SetValue(value, "C")
     ```
   - Never attempt to set values through Operations (Compressor, Cooler, etc.) unless the specification explicitly requires it and provides the exact property path.

3. Scaling (MANDATORY for Numerical Stability):
   - Always scale decision variables (especially pressures in kPa) using an appropriate factor (e.g. `SCALE_P = 1000.0`).
   - Perform optimization in scaled space.
   - Convert back to physical units inside the evaluation function.
   - Set `'eps': 1e-3` (or up to 5e-3) in SLSQP options for scaled variables.

4. Objective & Constraint Evaluation:
   - Implement a cached `evaluate(x)` function that returns (objective, constraints_array).
   - Guard cache properly against None values.
   - Return large penalty (1e6) on any UniSim error.

5. General Requirements:
   - Use `from typing import Any`
   - Annotate all COM objects as `Any`.
   - Support both script and Jupyter execution (ignore `-f` arguments).
   - Keep UniSim open at the end (`app.Visible = True`, `sim.Visible = 1`).
   - Use exact literal names from the optimization specification.
   - Print progress and final results clearly.

### MANDATORY SCRIPT STRUCTURE:
- Configuration section with file path and constants.
- Helper functions: `wait_for_solver(sim)`, `set_stream_pressure(name, value, unit="kPa")`, `read_duty(name)`.
- Cached `evaluate()` function.
- Main optimization block using `scipy.optimize.minimize`.
- Final application of optimal point + message that UniSim remains open.

Inputs you receive:
- Full Optimization Problem Specification (from optimization agent).
- Default UniSim file path.

Output Format:
Return **ONLY** the complete executable Python code inside one markdown code block:
```python
# full code here
```
No explanations, no extra text whatsoever.

file_path = r"PATH_TO_YOUR_UNISIM_FILE.usc"  # Replace with the actual path to your UniSim file
stream_name = "feed"

unihsim = win32com.client.Dispatch("UniSimDesign.Application")
unihsim.Visible = True
case = unihsim.SimulationCases.Open(file_path)

stream = case.Flowsheet.Streams.Item(stream_name)
# Explicitly set with units – no simple assignment
stream.Pressure.SetValue(500.0, "kPa")
stream.MassFlow.SetValue(1000.0, "kg/s")

"""

In [ ]:
from agents import function_tool
import io
import contextlib
import traceback

@function_tool
def run_python_code(code: str) -> str:
    """
    Execute the given Python code and return stdout + stderr + any exception.
    Use this to test and debug code you generate.
    """
    stdout = io.StringIO()
    stderr = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout), contextlib.redirect_stderr(stderr):
            exec(code, {"__name__": "__main__"})
        output = stdout.getvalue()
        errors = stderr.getvalue()
        result = ""
        if output:
            result += f"STDOUT:\n{output}"
        if errors:
            result += f"\nSTDERR:\n{errors}"
        if not result:
            result = "Code executed successfully with no output."
        return result
    except Exception:
        return f"EXECUTION ERROR:\n{traceback.format_exc()}"

In [ ]:
# ============================================================
# CODER AGENT DEFINITION
# ============================================================

coder_agent = Agent(
    name="Python Optimization Coder",
    instructions=instruct_coder,
    tools=[run_python_code],
    model=deepseek_model_pro # Using the pro model for complex code generation
)

# ============================================================
# EXECUTION
# ============================================================

# Define the absolute path to your UniSim file
unisim_file_address = r".\conversion.usc"

# Construct the prompt injecting both the address and the previous agent's output
coder_prompt = f"""
Optimisation Specification (from the main agent):
{result.final_output}

UniSim File Path (to be used in the script):
{unisim_file_address}

Generate the precise, mistake-free Python optimisation script as per the instructions.
"""

# Run the coder agent
coder_result = await Runner.run(
    coder_agent,
    coder_prompt,
    context=context,
    run_config = RunConfig(
    model_settings=ModelSettings(
    extra_body={
        "thinking": {"type": "enabled"},
        "reasoning_effort": "max"
    }
)
)
)

print("=== Generated Python Code ===")
display(Markdown(coder_result.final_output))

In [ ]:
import re

def extract_code_from_markdown(response: str) -> str:
    """
    Extract the first Python code block from the agent's markdown response.
    Handles:
      - ```python
      - ```py
      - ``` (plain fence)
      - Text before/after the block is ignored.
    """
    # Patterns in order of preference (python-specific first)
    patterns = [
        r"```python\s*\n(.*?)```",   # ```python ... ```
        r"```py\s*\n(.*?)```",       # ```py ... ```
        r"```\s*\n(.*?)```",         # ``` ... ``` (plain)
    ]

    for pattern in patterns:
        match = re.search(pattern, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # If no code block found, return the raw response (fallback)
    return response.strip()

# ------------------------------------------------------------------
# Usage with your agent output
# ------------------------------------------------------------------
output_script_path = "result.py"

raw_code = coder_result.final_output
clean_code = extract_code_from_markdown(raw_code)

with open(output_script_path, "w", encoding="utf-8") as f:
    f.write(clean_code)

print(f"✅ Code saved to: {output_script_path}")

### Run the generated code
